# Transformer Attention Mechanisms

This notebook explores attention mechanisms used in Large Language Models (LLMs).

## Contents
1. Scaled Dot-Product Attention
2. Multi-Head Attention
3. Grouped-Query Attention (GQA)
4. Rotary Position Embeddings (RoPE)
5. Gradient Flow in Transformers

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
sys.path.append('../src')

from attention import ScaledDotProductAttention, MultiHeadAttention
from attention import GroupedQueryAttention, CausalSelfAttention

print(f"PyTorch version: {torch.__version__}")
print("Libraries imported successfully!")

## 1. Scaled Dot-Product Attention

The foundation of transformer architectures:

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Where:
- $Q$ (queries): what we're looking for
- $K$ (keys): what each position offers
- $V$ (values): the actual information
- $d_k$: dimension of keys (scaling factor)

In [ ]:
# Create example input
batch_size = 1
n_heads = 1
seq_len = 8
d_k = 64

# Random Q, K, V
Q = torch.randn(batch_size, n_heads, seq_len, d_k)
K = torch.randn(batch_size, n_heads, seq_len, d_k)
V = torch.randn(batch_size, n_heads, seq_len, d_k)

# Apply attention
attention = ScaledDotProductAttention(dropout=0.0)
output, attention_weights = attention(Q, K, V)

print(f"Input shapes:")
print(f"  Q: {Q.shape}")
print(f"  K: {K.shape}")
print(f"  V: {V.shape}")
print(f"\nOutput shapes:")
print(f"  Output: {output.shape}")
print(f"  Attention weights: {attention_weights.shape}")

# Visualize attention weights
plt.figure(figsize=(8, 7))
sns.heatmap(attention_weights[0, 0].detach().numpy(), 
            cmap='viridis', annot=True, fmt='.2f', cbar=True,
            xticklabels=range(seq_len), yticklabels=range(seq_len))
plt.xlabel('Key/Value Position', fontsize=11)
plt.ylabel('Query Position', fontsize=11)
plt.title('Attention Weights Matrix', fontsize=13)
plt.show()

# Verify softmax property (rows sum to 1)
row_sums = attention_weights[0, 0].sum(dim=1)
print(f"\nRow sums (should all be 1.0): {row_sums}")

## 2. Multi-Head Attention

$$\text{MultiHead}(Q, K, V) = \text{Concat}(\text{head}_1, ..., \text{head}_h)W^O$$

where:

$$\text{head}_i = \text{Attention}(QW_i^Q, KW_i^K, VW_i^V)$$

**Purpose**: Allow model to attend to different representation subspaces

In [ ]:
# Parameters
d_model = 512
n_heads = 8
seq_len = 10
batch_size = 2

# Create input
x = torch.randn(batch_size, seq_len, d_model)

# Multi-head attention
mha = MultiHeadAttention(d_model, n_heads, dropout=0.1)

# Forward pass
output, attention_weights = mha(x, x, x)

print(f"Configuration:")
print(f"  d_model: {d_model}")
print(f"  n_heads: {n_heads}")
print(f"  d_k (per head): {d_model // n_heads}")
print(f"\nShapes:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}")
print(f"  Attention weights: {attention_weights.shape}")
print(f"\nParameters: {sum(p.numel() for p in mha.parameters()):,}")

In [ ]:
# Visualize attention patterns for all heads
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

# Take first batch item
attn = attention_weights[0].detach().numpy()

for i in range(n_heads):
    sns.heatmap(attn[i], cmap='viridis', cbar=True, ax=axes[i],
                xticklabels=False, yticklabels=False)
    axes[i].set_title(f'Head {i+1}', fontsize=11)

plt.suptitle('Attention Patterns Across All Heads', fontsize=14, y=0.995)
plt.tight_layout()
plt.show()

print("\nNote: Different heads learn different attention patterns!")

## 3. Grouped-Query Attention (GQA)

Used in modern LLMs (LLaMA 2/3, Gemma 2/3, Qwen).

**Idea**: Share key and value projections across multiple query heads.

- **MHA**: Each head has separate Q, K, V
- **GQA**: Multiple Q heads share the same K, V
- **MQA**: All Q heads share a single K, V

**Benefits**:
- Reduced memory for KV cache (important for long contexts)
- Faster inference
- Minimal quality loss

In [ ]:
# Compare MHA vs GQA
d_model = 512
n_heads = 8
n_kv_heads = 2  # GQA: 4 query heads per KV head

x = torch.randn(2, 10, d_model)

# Multi-Head Attention
mha = MultiHeadAttention(d_model, n_heads)
mha_output, _ = mha(x, x, x)

# Grouped-Query Attention
gqa = GroupedQueryAttention(d_model, n_heads, n_kv_heads)
gqa_output = gqa(x, x, x)

print("Multi-Head Attention (MHA):")
print(f"  Query heads: {n_heads}")
print(f"  KV heads: {n_heads}")
print(f"  Parameters: {sum(p.numel() for p in mha.parameters()):,}")

print("\nGrouped-Query Attention (GQA):")
print(f"  Query heads: {n_heads}")
print(f"  KV heads: {n_kv_heads}")
print(f"  Ratio: {n_heads // n_kv_heads}:1")
print(f"  Parameters: {sum(p.numel() for p in gqa.parameters()):,}")

mha_params = sum(p.numel() for p in mha.parameters())
gqa_params = sum(p.numel() for p in gqa.parameters())
reduction = (1 - gqa_params / mha_params) * 100

print(f"\nParameter reduction: {reduction:.1f}%")
print(f"Output shapes match: {mha_output.shape == gqa_output.shape}")

## 4. Causal Self-Attention with RoPE

**Causal masking**: Prevents attending to future positions (for autoregressive generation)

**RoPE**: Rotary Position Embeddings
- Encodes position through rotation
- Better extrapolation to longer sequences
- Used in most modern LLMs

In [ ]:
# Causal self-attention
d_model = 256
n_heads = 4
seq_len = 8

x = torch.randn(1, seq_len, d_model)

# With RoPE
causal_attn = CausalSelfAttention(d_model, n_heads, use_rope=True)
output = causal_attn(x)

print(f"Input shape: {x.shape}")
print(f"Output shape: {output.shape}")
print(f"Position encoding: RoPE (Rotary Position Embeddings)")
print(f"\nCausal mask ensures tokens only attend to past positions.")

## 5. Gradient Flow in Attention

**Why attention helps gradient flow:**

1. **Direct paths**: Skip connections provide direct gradient paths
2. **Softmax gradients**: Relatively smooth compared to RNN gates
3. **Multi-head**: Multiple gradient paths in parallel

**Gradient computation** (simplified):

$$\frac{\partial \text{Attention}}{\partial Q} \propto \text{softmax}(\text{scores}) \cdot V$$

The gradient doesn't vanish across long sequences like in RNNs!

In [ ]:
# Demonstrate gradient flow
d_model = 128
n_heads = 4
seq_len = 32

# Create simple model
x = torch.randn(1, seq_len, d_model, requires_grad=True)
mha = MultiHeadAttention(d_model, n_heads, dropout=0.0)

# Forward pass
output, _ = mha(x, x, x)

# Compute loss (sum of outputs)
loss = output.sum()

# Backward pass
loss.backward()

# Check gradient norms at different positions
grad_norms = torch.norm(x.grad, dim=-1).squeeze()

plt.figure(figsize=(12, 5))
plt.plot(grad_norms.detach().numpy(), 'b-', linewidth=2, marker='o')
plt.xlabel('Position in Sequence', fontsize=12)
plt.ylabel('Gradient Norm', fontsize=12)
plt.title('Gradient Flow Across Sequence Positions', fontsize=13)
plt.grid(True, alpha=0.3)
plt.axhline(y=grad_norms.mean().item(), color='r', linestyle='--', 
           linewidth=2, label=f'Mean: {grad_norms.mean():.4f}')
plt.legend(fontsize=10)
plt.show()

print(f"Gradient statistics:")
print(f"  Mean gradient norm: {grad_norms.mean():.6f}")
print(f"  Std gradient norm: {grad_norms.std():.6f}")
print(f"  Min gradient norm: {grad_norms.min():.6f}")
print(f"  Max gradient norm: {grad_norms.max():.6f}")
print(f"\nNote: Gradients remain relatively uniform across all positions!")
print("This is a key advantage over RNNs where gradients vanish for distant positions.")

## Key Takeaways

1. **Scaled dot-product attention** is the foundation:
   - Scaling by $\sqrt{d_k}$ prevents saturation
   - Softmax creates probability distribution
   - Weighted sum of values produces output

2. **Multi-head attention** provides:
   - Multiple representation subspaces
   - Richer attention patterns
   - Parallel gradient paths

3. **Grouped-Query Attention** optimizes:
   - Memory usage (KV cache)
   - Inference speed
   - Maintains quality

4. **Gradient flow** in transformers:
   - Better than RNNs for long sequences
   - Skip connections help
   - Still need gradient clipping for very deep models

5. **Modern LLMs** use:
   - GQA for efficiency
   - RoPE for position encoding
   - Causal masking for generation
   - RMSNorm for stability